# C13 — Modèles de production : entraînement final, calibration et poids safetensors

Ce notebook prépare les trois modèles de `configs/production.yaml` :

1. **émotions, tête whisper-base** : la tête finale de C11 (graine 42), telle quelle ;
   température T réglée sur la validation, ECE mesurée sur le test ;
2. **événements, tête whisper-base** : nouvelle tête entraînée sur les **5 folds** d'ESC-50,
   avec la médiane des réglages des 5 tours de C11 ; T = moyenne des 5 températures réglées
   chacune sur le fold de test de son tour ;
3. **événements, CRNN** : même principe, entraînement sur les 5 folds sur GPU, T issue des
   prédictions des 5 tours de C9.

Chaque modèle est écrit dans `/kaggle/working/production/<identifiant>/` : `model.safetensors`
(poids, sans pickle) et `config.json` (classes, standardisation, T, encodeur, empreinte des
features). La publication sur Hugging Face se fait dans un second notebook.

## a. Démarrage

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. GPU et réglages communs

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import torch

from echolens.config import CONFIG_DIR, REPO_ROOT
from echolens.core.data.class_labels import load_class_labels
from echolens.core.inference import load_inference_config
from echolens.core.models.production_config import EncoderReference
from echolens.training.embeddings import load_embeddings_config
from echolens.training.evaluation import EvaluationConfig
from echolens.training.heads import load_heads_config
from echolens.training.production import load_production_plan

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4 du notebook.")
device = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(device)
print("GPU :", GPU_NAME, "· torch", torch.__version__)
NOTEBOOK_START = time.time()

commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
plan = load_production_plan()
heads_config = load_heads_config()
evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / "evaluation.yaml")
inference_config = load_inference_config()
labels = load_class_labels()
emotion_ids = [class_label.id for class_label in labels.emotions]
esc50_ids = [class_label.id for class_label in labels.esc50_classes]
RESULTS_DIR = REPO_ROOT / "training" / "results"
OUTPUT_DIR = Path("/kaggle/working/production")

# Encodeur Whisper base de C10, à sa révision figée (configs/embeddings.yaml).
embeddings_config = load_embeddings_config()
for encoder in embeddings_config.encoders:
    if encoder.id == "whisper-base":
        whisper = encoder
encoder_reference = EncoderReference(
    checkpoint=whisper.checkpoint,
    revision=whisper.revision,
    sample_rate=embeddings_config.sample_rate,
    samples_per_output_frame=whisper.samples_per_output_frame,
    max_output_frames=whisper.max_output_frames,
)
print("commit", commit, "· encodeur", encoder_reference.checkpoint, encoder_reference.revision)

## c. Entrées

Sorties attachées au notebook : `echolens-c11-family-b-heads` (têtes et probabilités de C11),
`echolens-c9-family-a-esc50-crnn` (prédictions des 5 tours du CRNN), `echolens-c9-family-a-esc50`
(log-Mel d'ESC-50), et le dataset `echolens-emb-whisper-base` (moyennes par couche de C10).

In [ ]:
from echolens.training.production import RoundPredictions

INPUT_DIR = Path("/kaggle/input")


def find_one(pattern: str) -> Path:
    # Un seul fichier doit correspondre ; on écarte la version « avec remplissage » de C10.
    matches = [path for path in sorted(INPUT_DIR.glob(pattern)) if "padded" not in path.parts]
    if len(matches) != 1:
        raise RuntimeError(f"{len(matches)} fichiers pour {pattern} : il en faut exactement un")
    return matches[0]


def c11_predictions(path: Path) -> RoundPredictions:
    with np.load(path) as content:
        return RoundPredictions(y_true=content["targets"], probabilities=content["probabilities"])


def c9_predictions(path: Path) -> RoundPredictions:
    with np.load(path) as content:
        return RoundPredictions(
            y_true=content["test_labels"], probabilities=content["seed42_probabilities"]
        )


EMOTION_DIR = find_one("**/c11/emotions/whisper-base/head_seed42.pt").parent
ESC50_HEADS_DIR = find_one("**/c11/esc50/whisper-base/fold1/test_seed42.npz").parent.parent
# Prédictions des 5 tours du CRNN sur la courbe complète (PR #25) : session « -crnn ». Le
# kernel de C9 attaché pour le log-Mel contient aussi des prédictions du premier calcul
# (patience 15), qu'il ne faut pas prendre.
CRNN_ARRAYS_DIR = find_one("**/echolens-c9-family-a-esc50-crnn/c9/arrays/crnn_fold1.npz").parent
LOG_MELS_PATH = find_one("**/esc50_log_mels.npz")
EMBEDDINGS_DIR = find_one("**/esc50/embeddings.npy").parent
for name, path in [
    ("émotions C11", EMOTION_DIR),
    ("ESC-50 C11", ESC50_HEADS_DIR),
    ("CRNN C9", CRNN_ARRAYS_DIR),
    ("log-Mel", LOG_MELS_PATH),
    ("représentations", EMBEDDINGS_DIR),
]:
    print(f"{name} : {path}")
summary = {"commit": commit, "gpu": GPU_NAME, "torch_version": torch.__version__, "models": {}}

## d. Émotions : tête whisper-base de C11, calibrée sur la validation

La tête finale de la graine 42 a choisi sa meilleure époque sur la validation (C11). On la
reprend telle quelle, avec ses statistiques de standardisation. T se règle sur ses prédictions
de validation ; l'ECE se mesure sur le test (1 324 extraits jamais vus), avant et après.

In [ ]:
from echolens.core.models.heads import LayerWeightedHead
from echolens.core.models.production_config import (
    Calibration,
    HeadShape,
    ProductionModelConfig,
    TrainingRecord,
)
from echolens.core.models.storage import save_production_model
from echolens.training.production import validation_calibration

planned = plan.model("echolens-emotion-whisper-base")
emotion_text = (RESULTS_DIR / "c11_family_b_emotions.json").read_text(encoding="utf-8")
emotion_results = json.loads(emotion_text)
if emotion_results["classes"] != emotion_ids:
    raise RuntimeError("Les classes de C11 ne sont pas celles de configs/labels.yaml")
chosen = emotion_results["models"]["whisper-base"]["chosen"]
dropout = chosen["setting"]["dropout"]
learning_rate = chosen["setting"]["learning_rate"]

# Étape 1 : la tête de C11 et ses statistiques (fichier torch de notre propre notebook C11).
with np.load(EMOTION_DIR / "standardization.npz") as stats:
    emotion_mean = stats["mean"]
    emotion_std = stats["std"]
layer_count, dimension = emotion_mean.shape
shape = HeadShape(
    layer_count=layer_count,
    dimension=dimension,
    hidden_size=heads_config.hidden_size,
    dropout=dropout,
)
emotion_head = LayerWeightedHead(
    layer_count, dimension, shape.hidden_size, len(emotion_ids), dropout
)
state = torch.load(EMOTION_DIR / "head_seed42.pt", map_location="cpu", weights_only=True)
emotion_head.load_state_dict(state, strict=True)
emotion_head.eval()

# Étape 2 : T sur la validation, ECE sur le test.
validation = c11_predictions(EMOTION_DIR / "validation_seed42.npz")
test = c11_predictions(EMOTION_DIR / "test_seed42.npz")
outcome = validation_calibration(validation, test, evaluation_config)

emotion_config = ProductionModelConfig(
    model_id=planned.id,
    task="emotion",
    architecture="layer_weighted_head",
    labels=emotion_ids,
    standardization_mean=emotion_mean.tolist(),
    standardization_std=emotion_std.tolist(),
    calibration=Calibration(
        temperature=outcome.temperature,
        method="T réglée sur la validation (graine 42), ECE mesurée sur le test",
        ece_before=outcome.ece_before,
        ece_after=outcome.ece_after,
    ),
    training=TrainingRecord(
        learning_rate=learning_rate,
        dropout=dropout,
        epochs=chosen["best_epoch"],
        seed=42,
        data=planned.data,
        commit=emotion_results["commit"],
    ),
    window_s=inference_config.emotion.max_window_s,
    head=shape,
    encoder=encoder_reference,
    license=planned.license,
)
save_production_model(emotion_head, emotion_config, OUTPUT_DIR / planned.id)
summary["models"][planned.id] = emotion_config.calibration.model_dump()
ece_text = f"ECE {outcome.ece_before:.4f} → {outcome.ece_after:.4f}"
print(f"{planned.id} : T = {outcome.temperature:.3f}, {ece_text}")

## e. Événements : tête whisper-base entraînée sur les 5 folds

Réglage = médiane des réglages retenus aux 5 tours de C11 (taux, dropout, époques). Les
statistiques de standardisation sont calculées sur les 2 000 extraits (tout l'entraînement du
modèle de production, R4). T = moyenne des 5 températures de C11 ; ECE hors fold.

In [ ]:
from echolens.core.data.manifest import load_manifest
from echolens.training.head_training import train_for_epochs
from echolens.training.heads import (
    HeadSetting,
    LabeledSet,
    esc50_targets_and_folds,
    standardization_stats,
    standardize_set,
)
from echolens.training.production import median_value, out_of_fold_calibration

planned = plan.model("echolens-sound-event-whisper-base")
esc50_results = json.loads((RESULTS_DIR / "c11_family_b_esc50.json").read_text(encoding="utf-8"))
per_fold = esc50_results["models"]["whisper-base"]["per_fold"]
learning_rate = median_value([fold["learning_rate"] for fold in per_fold])
dropout = median_value([fold["dropout"] for fold in per_fold])
epochs = int(median_value([fold["epochs"] for fold in per_fold]))
print("Réglage médian :", learning_rate, dropout, epochs)

# Étape 1 : les 2 000 extraits, dans l'ordre du manifeste versionné.
rows = load_manifest(REPO_ROOT / "data" / "manifests" / "esc50.csv")
embeddings = np.load(EMBEDDINGS_DIR / "embeddings.npy")
index_paths = (EMBEDDINGS_DIR / "index.csv").read_text(encoding="utf-8").splitlines()[1:]
if [line.split(",")[1] for line in index_paths] != [row.path for row in rows]:
    raise RuntimeError("Les représentations ne suivent pas l'ordre du manifeste")
targets, _ = esc50_targets_and_folds(rows)

# Étape 2 : standardisation sur les 2 000 extraits, puis entraînement (graine 42).
event_stats = standardization_stats(embeddings)
train = standardize_set(LabeledSet(features=embeddings, targets=targets), event_stats)
start = time.time()
event_head = train_for_epochs(
    train,
    HeadSetting(learning_rate=learning_rate, dropout=dropout),
    heads_config,
    len(esc50_ids),
    plan.seed,
    epochs,
)
event_head.eval()
head_seconds = time.time() - start

# Étape 3 : température hors fold, à partir des 5 tours de C11.
rounds = [
    c11_predictions(ESC50_HEADS_DIR / f"fold{fold}" / "test_seed42.npz") for fold in range(1, 6)
]
outcome = out_of_fold_calibration(rounds, evaluation_config)

layer_count, dimension = event_stats.mean.shape
event_head_config = ProductionModelConfig(
    model_id=planned.id,
    task="sound_event",
    architecture="layer_weighted_head",
    labels=esc50_ids,
    standardization_mean=event_stats.mean.tolist(),
    standardization_std=event_stats.std.tolist(),
    calibration=Calibration(
        temperature=outcome.temperature,
        method="moyenne des T des 5 tours de C11 ; ECE hors fold (T des 4 autres tours)",
        ece_before=outcome.ece_before,
        ece_after=outcome.ece_after,
        round_temperatures=outcome.round_temperatures,
    ),
    training=TrainingRecord(
        learning_rate=learning_rate,
        dropout=dropout,
        epochs=epochs,
        seed=plan.seed,
        data=planned.data,
        commit=commit,
    ),
    window_s=inference_config.sound_event.window_s,
    head=HeadShape(
        layer_count=layer_count,
        dimension=dimension,
        hidden_size=heads_config.hidden_size,
        dropout=dropout,
    ),
    encoder=encoder_reference,
    license=planned.license,
)
save_production_model(event_head, event_head_config, OUTPUT_DIR / planned.id)
summary["models"][planned.id] = event_head_config.calibration.model_dump()
summary["models"][planned.id]["train_seconds"] = head_seconds
round_text = [round(temperature, 3) for temperature in outcome.round_temperatures]
ece_text = f"ECE {outcome.ece_before:.4f} → {outcome.ece_after:.4f}"
print(f"{planned.id} : T = {outcome.temperature:.3f} (tours {round_text}), {ece_text}")
print(f"entraînement : {head_seconds:.0f} s")

## f. Événements : CRNN entraîné sur les 5 folds (GPU)

Réglage = médiane des réglages des 5 tours de C9 (courbe complète). Statistiques par bande sur
les 2 000 log-Mel (R4). T = moyenne des 5 températures réglées sur les folds de test de C9.

In [ ]:
from echolens.core.models.production_config import file_sha256
from echolens.training.family_a import load_family_a_config, load_log_mels
from echolens.training.family_a_models import LabeledSpectrograms, train_fixed_epochs

planned = plan.model("echolens-sound-event-crnn")
crnn_results = json.loads((RESULTS_DIR / "esc50_family_a_crnn.json").read_text(encoding="utf-8"))
learning_rate = median_value(
    [round_details["chosen_learning_rate"] for round_details in crnn_results["per_round"]]
)
epochs = int(
    median_value([round_details["chosen_epochs"] for round_details in crnn_results["per_round"]])
)
family_a_config = load_family_a_config()
print("Réglage médian :", learning_rate, epochs)

# Étape 1 : les 2 000 log-Mel de C9, déjà calculés (R6).
log_mel_set = load_log_mels(LOG_MELS_PATH)
if log_mel_set.paths != [row.path for row in rows]:
    raise RuntimeError("Les log-Mel ne suivent pas l'ordre du manifeste")
train = LabeledSpectrograms(log_mels=log_mel_set.log_mels, labels=log_mel_set.labels)

# Étape 2 : entraînement final sur les 5 folds, graine 42.
start = time.time()
trained = train_fixed_epochs(
    "crnn", train, len(esc50_ids), learning_rate, epochs, plan.seed, family_a_config, device
)
crnn_seconds = time.time() - start

# Étape 3 : température hors fold, à partir des 5 tours de C9.
rounds = [c9_predictions(CRNN_ARRAYS_DIR / f"crnn_fold{fold}.npz") for fold in range(1, 6)]
outcome = out_of_fold_calibration(rounds, evaluation_config)

crnn_config = ProductionModelConfig(
    model_id=planned.id,
    task="sound_event",
    architecture="crnn",
    labels=esc50_ids,
    standardization_mean=trained.statistics.mean.tolist(),
    standardization_std=trained.statistics.std.tolist(),
    calibration=Calibration(
        temperature=outcome.temperature,
        method="moyenne des T des 5 tours de C9 ; ECE hors fold (T des 4 autres tours)",
        ece_before=outcome.ece_before,
        ece_after=outcome.ece_after,
        round_temperatures=outcome.round_temperatures,
    ),
    training=TrainingRecord(
        learning_rate=learning_rate,
        dropout=family_a_config.network.dropout,
        epochs=epochs,
        seed=plan.seed,
        data=planned.data,
        commit=commit,
    ),
    window_s=inference_config.sound_event.window_s,
    network=family_a_config.network,
    features_sha256=file_sha256(CONFIG_DIR / "features.yaml"),
    license=planned.license,
)
save_production_model(trained.network, crnn_config, OUTPUT_DIR / planned.id)
summary["models"][planned.id] = crnn_config.calibration.model_dump()
summary["models"][planned.id]["train_seconds"] = crnn_seconds
round_text = [round(temperature, 3) for temperature in outcome.round_temperatures]
ece_text = f"ECE {outcome.ece_before:.4f} → {outcome.ece_after:.4f}"
print(f"{planned.id} : T = {outcome.temperature:.3f} (tours {round_text}), {ece_text}")
print(f"entraînement : {crnn_seconds / 60:.1f} min")

## g. Contrôle : chaque modèle se recharge depuis ses fichiers et redonne les mêmes sorties

On reconstruit chaque réseau à partir de `config.json`, on charge `model.safetensors` (sans
pickle), puis on compare ses sorties à celles du réseau en mémoire, sur les mêmes entrées.

In [ ]:
from echolens.core.models.family_a import SpectrogramCrnn
from echolens.core.models.production_config import CONFIG_FILE, WEIGHTS_FILE, load_production_config
from echolens.core.models.storage import load_weights

generator = torch.Generator().manual_seed(0)
for network, model_id in [
    (emotion_head, "echolens-emotion-whisper-base"),
    (event_head, "echolens-sound-event-whisper-base"),
    (trained.network, "echolens-sound-event-crnn"),
]:
    folder = OUTPUT_DIR / model_id
    reloaded_config = load_production_config(folder / CONFIG_FILE)
    network = network.cpu().eval()
    if reloaded_config.architecture == "crnn":
        fresh = SpectrogramCrnn(len(reloaded_config.labels), reloaded_config.network)
        inputs = torch.randn(2, 64, 501, generator=generator)
    else:
        shape = reloaded_config.head
        fresh = LayerWeightedHead(
            shape.layer_count,
            shape.dimension,
            shape.hidden_size,
            len(reloaded_config.labels),
            shape.dropout,
        )
        inputs = torch.randn(4, shape.layer_count, shape.dimension, generator=generator)
    reloaded = load_weights(fresh, folder / WEIGHTS_FILE)
    with torch.no_grad():
        expected = network(inputs)
        actual = reloaded(inputs)
    if reloaded_config.architecture == "crnn":
        expected = expected.logits
        actual = actual.logits
    torch.testing.assert_close(actual, expected, rtol=0, atol=0)
    size_mb = (folder / WEIGHTS_FILE).stat().st_size / 1e6
    parameters = sum(parameter.numel() for parameter in fresh.parameters())
    summary["models"][model_id]["parameters"] = parameters
    summary["models"][model_id]["weights_mb"] = round(size_mb, 3)
    print(f"{model_id} : rechargé à l'identique · {parameters} paramètres · {size_mb:.2f} Mo")

## h. Résumé et temps

In [ ]:
import shutil

summary["notebook_minutes"] = (time.time() - NOTEBOOK_START) / 60
summary_path = Path("/kaggle/working/c13_production_summary.json")
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(sorted(str(path.relative_to(OUTPUT_DIR)) for path in OUTPUT_DIR.glob("*/*")))